# MAKE SURE TO CHECK AND CHANGE THIS CELL BEFORE EVERY RUN

## CURRENTLY RUNNING: WHISPER LARGE V3 TURBO

In [ ]:
%%capture
!pip install transformers "datasets<3.0" peft accelerate bitsandbytes evaluate jiwer trl pydub soundfile librosa

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
secret_value_1 = user_secrets.get_secret("HF_WRITE")

In [ ]:
from huggingface_hub import login

login(secret_value_0)

In [ ]:
#final_data check
from datasets.load import load_dataset

dataset = load_dataset("abhi8799/Punjabi-STT_Combined")

In [ ]:
from datasets import Audio

# 1. Critical Step: Resample audio to 16kHz
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

In [ ]:
dataset = dataset.remove_columns(['token_count'])

In [ ]:
# from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

# #Set the model
# model_id = "openai/whisper-large-v3-turbo"

# # Load preprocessors
# feature_extractor = WhisperFeatureExtractor.from_pretrained(model_id)
# # Set language to punjabi and task to transcribe
# tokenizer = WhisperTokenizer.from_pretrained(model_id, language="punjabi", task="transcribe")
# processor = WhisperProcessor.from_pretrained(model_id, language="punjabi", task="transcribe")

# max_label_length = 448

# def prepare_dataset(batch):
#     # Load and resample audio data
#     audio = batch["audio"]

#     # Compute log-Mel spectrogram input features
#     batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

#     # Encode target text to label ids
#     batch["labels"] = tokenizer(batch["sentence"]).input_ids
#     return batch

# ## When Dataset showed audio corrupted
# # def prepare_dataset_batched(batch):
# #     # Loop manually over the batch items to find the broken file
# #     for idx, audio_item in enumerate(batch["audio"]):
# #         try:
# #             # Touch or read a property to force torchcodec to decode it
# #             _ = audio_item["array"] 
# #         except Exception as e:
# #             print(f"💥 Corrupted audio found! Batch index: {idx}")
# #             print(f"Problematic Item Meta: {audio_item}")
# #             raise e
            
# #     # --- Your original mapping logic here ---
# #     # ...
# #     return batch

# # # Run with num_proc removed or set to 1 for debugging
# # dataset = dataset.map(
# #     prepare_dataset_batched,
# #     batched=True,
# #     num_proc=1, # <--- CRITICAL: Force single-process to see print statements
# # )

# # def prepare_dataset_batched(batch):
# #     # Process audio features
# #     input_features = [
# #         feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
# #         for audio in batch["audio"]
# #     ]

# #     # CRITICAL: Tokenize WITHOUT truncation so we can see the true token length
# #     labels = tokenizer(batch["sentence"]).input_ids

# #     valid_input_features = []
# #     valid_labels = []

# #     # Iterate and completely DROP any pair where text exceeds 448
# #     for feature, label in zip(input_features, labels):
# #         # if len(label) <= max_label_length:
# #             valid_input_features.append(feature)
# #             valid_labels.append(label)
# #         # If it's over 448, we skip it entirely to protect alignment!

# #     return {
# #         "input_features": valid_input_features,
# #         "labels": valid_labels
# #     }

# # Apply to your dataset
# # dataset = dataset.map(
# #     prepare_dataset_batched,
# #     batched=True,
# #     batch_size=100,
# #     remove_columns=dataset["train"].column_names,
# #     num_proc=4
# # )

# dataset = dataset.map(prepare_dataset, remove_columns=dataset["train1"].column_names, num_proc=125)

# # dataset = dataset.map(
# #     prepare_dataset, 
# #     batched=True,             
# #     batch_size=100,           
# #     remove_columns=dataset["train"].column_names, 
# #     num_proc=4,
# #     load_from_cache_file=False # Forces Hugging Face to overwrite the old 80-bin features
# # )
# # def chunk_dataset_to_10s(batch):
# #     new_input_features = []
# #     new_labels = []
    
# #     # Target length: 10 seconds of audio at 16kHz = 160,000 samples
# #     target_samples = 160000 
    
# #     for audio, sentence in zip(batch["audio"], batch["sentence"]):
# #         array = audio["array"]
# #         sr = audio["sampling_rate"]
        
# #         # Total samples in this 30s audio track
# #         total_samples = len(array)
        
# #         # Calculate how many 10s chunks we can get
# #         num_chunks = total_samples // target_samples
        
# #         if num_chunks == 0:
# #             # Keep it if it's already shorter than 10s
# #             chunks = [array]
# #         else:
# #             chunks = [array[i*target_samples : (i+1)*target_samples] for i in range(num_chunks)]
        
# #         # Tokenize the full sentence to split or distribute
# #         token_ids = tokenizer(sentence).input_ids
        
# #         # Approximate text splitting per chunk (simple uniform split)
# #         tokens_per_chunk = len(token_ids) // len(chunks)
        
# #         for idx, chunk_array in enumerate(chunks):
# #             # Create the 128-bin audio features for this specific chunk
# #             features = feature_extractor(chunk_array, sampling_rate=sr).input_features[0]
            
# #             # Slice the corresponding tokens for this chunk
# #             start_tok = idx * tokens_per_chunk
# #             end_tok = (idx + 1) * tokens_per_chunk if idx < len(chunks) - 1 else len(token_ids)
# #             chunk_labels = token_ids[start_tok:end_tok]
            
# #             # Ensure it's not empty and strictly under the safety ceiling
# #             if 0 < len(chunk_labels) <= 448:
# #                 new_input_features.append(features)
# #                 new_labels.append(chunk_labels)
                
# #     return {
# #         "input_features": new_input_features,
# #         "labels": new_labels
# #     }

# # # Apply the 10s chunker to your dataset (Make sure to run on the raw, unmapped dataset)
# # dataset = dataset.map(
# #     chunk_dataset_to_10s,
# #     batched=True,
# #     batch_size=50, # Smaller batch size to prevent VRAM spikes during array slicing
# #     remove_columns=dataset["train"].column_names,
# #     num_proc=1 # Keep it at 1 to prevent memory collating race conditions on Kaggle
# # )

# #ALready split: 840 + 361
# # Split into train and test sets (e.g., 90% train, 10% eval)
# # dataset = dataset["train"].train_test_split(test_size=0.1)

from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor
from datasets import concatenate_datasets

# Set the model
model_id = "openai/whisper-large-v3-turbo"

# Load preprocessors
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_id)
tokenizer = WhisperTokenizer.from_pretrained(model_id, language="punjabi", task="transcribe")
processor = WhisperProcessor.from_pretrained(model_id, language="punjabi", task="transcribe")

max_label_length = 448

def prepare_dataset(batch):
    try:
        # Load and resample audio data
        audio = batch["audio"]

        # Compute log-Mel spectrogram input features
        batch["input_features"] = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]

        # Encode target text to label ids
        batch["labels"] = tokenizer(batch["sentence"]).input_ids
        return batch
    except Exception as e:
        # Return None for corrupted audio — will be filtered out
        print(f"⚠️ Skipping corrupted audio sample: {e}")
        batch["input_features"] = None
        batch["labels"] = None
        return batch

# 1. Map across ALL splits in the dataset dictionary
# We use dataset["train1"].column_names assuming all splits share the same schema
dataset = dataset.map(
    prepare_dataset, 
    remove_columns=dataset["train1"].column_names, 
    num_proc=1
)

# Drop any corrupted audio samples that failed to decode
dataset = dataset.filter(lambda x: x["input_features"] is not None)
print(f"Dataset after filtering corrupted samples: {dataset}")

# 2. Combine train1 and train2 into a single 'train' split
dataset["train"] = concatenate_datasets([
    dataset["train1"], 
    dataset["train2"]
])

# 3. Clean up the individual train1 and train2 splits to save memory/structure
del dataset["train1"]
del dataset["train2"]

# Check your final structure (It will now have ['test', 'train'])
print(dataset)

In [ ]:
dataset

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int = 50258  # Default start token for Whisper models
    pad_token_id: int = -100

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # 1. Pad audio input features
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # 2. Pad text label features
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Mask padding with -100
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # Truncate BOS token if present
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        # --- THE FIX: MANUALLY COMPUTE DECODER INPUT IDS ---
        # Shift the labels to the right to construct the perfect initial decoder state
        shifted_labels = labels.new_zeros(labels.shape)
        shifted_labels[:, 1:] = labels[:, :-1].clone()
        shifted_labels[:, 0] = self.decoder_start_token_id
        
        # Replace any remaining -100 values in decoder inputs with the processor's true pad ID
        true_pad_token_id = self.processor.tokenizer.pad_token_id
        shifted_labels.masked_fill_(shifted_labels == -100, true_pad_token_id)
        
        batch["decoder_input_ids"] = shifted_labels

        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [ ]:
from transformers import WhisperForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch


model_id = "openai/whisper-large-v3-turbo"

# 1. Define the 8-bit quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True
)

# 2. Pass the config to the model initialization
model = WhisperForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quantization_config,  # Use this instead of load_in_8bit=True
    device_map="auto"  # Remove so that model utilizes both available GPUS
)

# Prepare model for PEFT int8 training quirks
model = prepare_model_for_kbit_training(model)

# Define LoRA Configuration
config = LoraConfig(
    r=16,                                # Dropped from 32 to 16 to reduce capacity (less overfitting)
    lora_alpha=32,                       # Scaled proportionally (r * 2)
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"],
    lora_dropout=0.1,                    # Increased from 0.05 to 0.1 to drop out more activations
    bias="none"
)

model = get_peft_model(model, config)
model.print_trainable_parameters()
# You will notice that < 2% of the parameters are actually trainable now!

In [ ]:
# # FOR COLAB
# from google.colab import userdata
# HF_WRITE = userdata.get('HF_DATASET_TOKEN')

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-large-v3-turbo-gurmukhi-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,                  # 1. LOWER LEARNING RATE to slow down overfitting
    warmup_steps=100,
    max_steps=1000,

    # --- STRICT EVAL & SAVE ALIGNMENT FOR EARLY STOPPING ---
    eval_strategy="steps",
    eval_steps=50,                       # Check validation loss every 50 steps
    save_strategy="steps",
    save_steps=50,                       # MUST match eval_steps
    save_total_limit=2,                  # Only keeps the best and most recent checkpoints to save space

    # --- REGULARIZATION ---
    weight_decay=0.05,                   # Adds L2 regularization penalty to weights
    label_smoothing_factor=0.1,          # Prevents the model from over-allocating confidence to tokens

    predict_with_generate=False,
    logging_steps=10,

    fp16=True,
    gradient_checkpointing=True,
    optim="adamw_bnb_8bit",

    # --- HUB SETTINGS ---
    push_to_hub=True,                    # Enable uploading to HF Hub
    hub_model_id="abhi8799/whisper-large-v3-turbo-gurmukhi-lora", # Target repo ID
    hub_strategy="every_save",           # Upload checkpoints as they are saved
    hub_token= secret_value_1,

    # --- CRITICAL ---
    load_best_model_at_end=True,         # Required for Early Stopping to pull the absolute lowest loss checkpoint
    metric_for_best_model="loss",        # Track loss
    greater_is_better=False              # Lower loss is better
)

# Fix configurations for generating text during evaluation
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    processing_class=processor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
# 1. Start training
trainer.train()

# 2. Push the final trained LoRA adapters and tokenizer configurations to the Hub
trainer.push_to_hub(tags="automatic-speech-recognition", commit_message = "Finally Commited. End of training")

If the model hasn't completely trained and stops midway, store and push current model which has been safely stored on HF hub

In [ ]:
# !unzip results.zip -d /kaggle/working/

In [ ]:
# from transformers import WhisperForConditionalGeneration, BitsAndBytesConfig
# from peft import PeftModel
# from huggingface_hub import login

# model_id = "openai/whisper-small" ###############################################################################

# # 1. Log in to your Hugging Face account
# # (Make sure to use a token with WRITE permissions)
# login(secret_value_1)#########################################################################################

# # 2. Re-define the quantization config just in case the notebook state cleared it
# quantization_config = BitsAndBytesConfig(load_in_8bit=True)

# # 3. Load the base model
# base_model = WhisperForConditionalGeneration.from_pretrained(
#     model_id,
#     quantization_config=quantization_config,
#     device_map="auto"
# )

# # 4. Point to your highest checkpoint number in Kaggle's directory
# # Change 'checkpoint-1000' to whatever your highest number is
# final_checkpoint_path = "/kaggle/working/whisper-small-gurmukhi-lora/checkpoint-1000" ####################################################

# # 5. Combine the base model with your trained LoRA weights
# model = PeftModel.from_pretrained(base_model, final_checkpoint_path)

# # 6. Push everything to the Hub securely
# repo_id = "abhi8799/whisper-small-gurmukhi-lora" ################################################################

# print("Uploading weights...")
# model.push_to_hub(repo_id, commit_message="End of fine-tuning on Gurmukhi dataset")

# print("Uploading processor...")
# processor.push_to_hub(repo_id)

# print("Done! Check your Hugging Face profile.")